# hypothesis tests

we have 3 questions to test:

1. do people buy more (more orders) on rainy or cold days?
2. do people spend more per order when the weather is bad?
3. are orders placed in bad weather cancelled more often?

for tests 1 and 2 we use a t-test. a t-test checks if the average of two groups is different enough that it probably isnt just random chance.

for test 3 we use a chi-square test. we use that one because we are comparing counts (how many cancelled vs not) across two groups, not averages.

in all tests: if p-value < 0.05 we say the result is statistically significant (unlikely to be random)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("cleaned/master.csv", parse_dates=["purchase_date"])
print("loaded:", df.shape)

loaded: (88675, 23)


## define bad vs good weather

we need to split orders into two groups for each test.

for rain: rainy = any rain at all (>0mm), dry = 0mm

for temperature: we use the median as the split point. below median = cold, above = hot.
using the median means we get two equal sized groups which is better for t-tests.

In [3]:
# rain: rainy vs dry
rainy = df[df['precip_total_mm'] > 0]
dry   = df[df['precip_total_mm'] == 0]

# temperature: cold (below median) vs hot (above median)
temp_median = df['temp_mean_c'].median()
cold = df[df['temp_mean_c'] <= temp_median]
hot  = df[df['temp_mean_c'] >  temp_median]

print('rainy orders:', len(rainy), '| dry orders:', len(dry))
print('cold orders:', len(cold), '| hot orders:', len(hot))
print('temp median used as split:', round(temp_median, 1), 'C')

rainy orders: 35806 | dry orders: 52869
cold orders: 44355 | hot orders: 44320
temp median used as split: 21.7 C


## test 1 - does weather affect how many orders are made?

we cant just count orders in rainy vs dry groups because there are different numbers of rainy and dry days.
instead we aggregate to the day level first (how many orders per day) then compare the two groups of days.

H0 (null hypothesis): the average number of daily orders is the same on rainy and dry days
H1 (alternative): they are different

In [4]:
# count orders per day, and record whether that day was rainy or dry
# we use the most common rain status across all orders that day
daily = df.groupby('purchase_date').agg(
    order_count  = ('order_id', 'count'),
    avg_rain     = ('precip_total_mm', 'mean'),
    avg_temp     = ('temp_mean_c', 'mean'),
    avg_spend    = ('total_price', 'mean')
).reset_index()

daily['is_rainy'] = daily['avg_rain'] > 0
daily['is_cold']  = daily['avg_temp'] <= daily['avg_temp'].median()

print(daily.shape)
daily.head(3)

(616, 7)


,purchase_date,order_count,avg_rain,avg_temp,avg_spend,is_rainy,is_cold
0,2016-09-04,1,1.0,26.708333,72.89,True,False
1,2016-09-05,1,24.6,14.275000,59.50,True,True
2,2016-09-15,1,0.0,23.775000,134.97,False,False


In [5]:
# split into two groups of days
orders_on_rainy_days = daily[daily['is_rainy'] == True]['order_count']
orders_on_dry_days   = daily[daily['is_rainy'] == False]['order_count']

print('avg orders on rainy days:', round(orders_on_rainy_days.mean(), 1))
print('avg orders on dry days:  ', round(orders_on_dry_days.mean(), 1))

# run the t-test
t_stat, p_value = stats.ttest_ind(orders_on_rainy_days, orders_on_dry_days)

print()
print('t-statistic:', round(t_stat, 4))
print('p-value:    ', round(p_value, 4))

if p_value < 0.05:
    print('result: SIGNIFICANT - rain does affect order volume (p < 0.05)')
else:
    print('result: NOT significant - no clear effect of rain on order volume (p >= 0.05)')

avg orders on rainy days: 144.6
avg orders on dry days:   8.7

t-statistic: 2.7548
p-value:     0.006
result: SIGNIFICANT - rain does affect order volume (p < 0.05)


In [6]:
# same test but for temperature (cold vs hot days)
orders_on_cold_days = daily[daily['is_cold'] == True]['order_count']
orders_on_hot_days  = daily[daily['is_cold'] == False]['order_count']

print('avg orders on cold days:', round(orders_on_cold_days.mean(), 1))
print('avg orders on hot days: ', round(orders_on_hot_days.mean(), 1))

t_stat2, p_value2 = stats.ttest_ind(orders_on_cold_days, orders_on_hot_days)

print()
print('t-statistic:', round(t_stat2, 4))
print('p-value:    ', round(p_value2, 4))

if p_value2 < 0.05:
    print('result: SIGNIFICANT - temperature does affect order volume (p < 0.05)')
else:
    print('result: NOT significant - no clear effect of temperature on order volume (p >= 0.05)')

avg orders on cold days: 144.8
avg orders on hot days:  143.1

t-statistic: 0.2569
p-value:     0.7973
result: NOT significant - no clear effect of temperature on order volume (p >= 0.05)


## test 2 - does weather affect how much people spend per order?

here we compare the average spend per order between rainy and dry days, and between cold and hot days.
this time we use individual orders (not daily aggregates) because each order has its own spend value.

H0: average spend is the same in good and bad weather
H1: average spend is different

In [7]:
# spend on rainy vs dry days
spend_rainy = rainy['total_price']
spend_dry   = dry['total_price']

print('avg spend on rainy days: R$', round(spend_rainy.mean(), 2))
print('avg spend on dry days:   R$', round(spend_dry.mean(), 2))

t_stat3, p_value3 = stats.ttest_ind(spend_rainy, spend_dry)

print()
print('t-statistic:', round(t_stat3, 4))
print('p-value:    ', round(p_value3, 4))

if p_value3 < 0.05:
    print('result: SIGNIFICANT - rain does affect spend per order (p < 0.05)')
else:
    print('result: NOT significant - no clear effect of rain on spend (p >= 0.05)')

avg spend on rainy days: R$ 138.05
avg spend on dry days:   R$ 138.91

t-statistic: -0.5967
p-value:     0.5507
result: NOT significant - no clear effect of rain on spend (p >= 0.05)


In [8]:
# spend on cold vs hot days
spend_cold = cold['total_price']
spend_hot  = hot['total_price']

print('avg spend on cold days: R$', round(spend_cold.mean(), 2))
print('avg spend on hot days:  R$', round(spend_hot.mean(), 2))

t_stat4, p_value4 = stats.ttest_ind(spend_cold, spend_hot)

print()
print('t-statistic:', round(t_stat4, 4))
print('p-value:    ', round(p_value4, 4))

if p_value4 < 0.05:
    print('result: SIGNIFICANT - temperature does affect spend per order (p < 0.05)')
else:
    print('result: NOT significant - no clear effect of temperature on spend (p >= 0.05)')

avg spend on cold days: R$ 133.43
avg spend on hot days:  R$ 143.7

t-statistic: -7.206
p-value:     0.0
result: SIGNIFICANT - temperature does affect spend per order (p < 0.05)


## test 3 - are orders placed in bad weather cancelled more? (the guilt question)

the idea is that people buy impulsively when they are stuck inside due to bad weather, then regret it and cancel.

for this we use a chi-square test. it works by building a table of counts:

```
             cancelled   not cancelled
rainy day       X              X
dry day         X              X
```

the test checks if the proportion of cancellations is the same in both rows or if it is genuinely different.

H0: cancellation rate is the same on rainy and dry days
H1: cancellation rate is different

In [9]:
# build the contingency table for rain
rainy_cancelled     = rainy['is_returned'].sum()
rainy_not_cancelled = len(rainy) - rainy_cancelled
dry_cancelled       = dry['is_returned'].sum()
dry_not_cancelled   = len(dry) - dry_cancelled

contingency_rain = [
    [rainy_cancelled,     rainy_not_cancelled],
    [dry_cancelled,       dry_not_cancelled]
]

print('contingency table (rain):')
print('             cancelled  not cancelled')
print('rainy days:  ', rainy_cancelled, '        ', rainy_not_cancelled)
print('dry days:    ', dry_cancelled, '          ', dry_not_cancelled)
print()
print('cancellation rate rainy:', round(rainy_cancelled / len(rainy) * 100, 3), '%')
print('cancellation rate dry:  ', round(dry_cancelled / len(dry) * 100, 3), '%')

contingency table (rain):
             cancelled  not cancelled
rainy days:   168          35638
dry days:     239            52630

cancellation rate rainy: 0.469 %
cancellation rate dry:   0.452 %


In [10]:
chi2, p_value5, dof, expected = stats.chi2_contingency(contingency_rain)

print('chi-square statistic:', round(chi2, 4))
print('p-value:             ', round(p_value5, 4))
print('degrees of freedom:  ', dof)

if p_value5 < 0.05:
    print('result: SIGNIFICANT - cancellation rate IS different on rainy vs dry days (p < 0.05)')
else:
    print('result: NOT significant - no clear difference in cancellation rate (p >= 0.05)')

chi-square statistic: 0.1022
p-value:              0.7492
degrees of freedom:   1
result: NOT significant - no clear difference in cancellation rate (p >= 0.05)


In [11]:
# same test but for temperature (cold vs hot)
cold_cancelled     = cold['is_returned'].sum()
cold_not_cancelled = len(cold) - cold_cancelled
hot_cancelled      = hot['is_returned'].sum()
hot_not_cancelled  = len(hot) - hot_cancelled

contingency_temp = [
    [cold_cancelled,  cold_not_cancelled],
    [hot_cancelled,   hot_not_cancelled]
]

print('contingency table (temperature):')
print('             cancelled  not cancelled')
print('cold days:   ', cold_cancelled, '         ', cold_not_cancelled)
print('hot days:    ', hot_cancelled, '          ', hot_not_cancelled)
print()
print('cancellation rate cold:', round(cold_cancelled / len(cold) * 100, 3), '%')
print('cancellation rate hot: ', round(hot_cancelled / len(hot) * 100, 3), '%')

chi2b, p_value6, dof2, expected2 = stats.chi2_contingency(contingency_temp)

print()
print('chi-square statistic:', round(chi2b, 4))
print('p-value:             ', round(p_value6, 4))

if p_value6 < 0.05:
    print('result: SIGNIFICANT - cancellation rate IS different on cold vs hot days (p < 0.05)')
else:
    print('result: NOT significant - no clear difference in cancellation rate (p >= 0.05)')

contingency table (temperature):
             cancelled  not cancelled
cold days:    238           44117
hot days:     169            44151

cancellation rate cold: 0.537 %
cancellation rate hot:  0.381 %

chi-square statistic: 11.3597
p-value:              0.0008
result: SIGNIFICANT - cancellation rate IS different on cold vs hot days (p < 0.05)


## summary of all results

In [12]:

print('HYPOTHESIS TEST RESULTS')

print()
print('TEST 1: does weather affect order VOLUME?')
print(f'  rain:        p = {round(p_value, 4)}  -> {"significant" if p_value < 0.05 else "not significant"}')
print(f'  temperature: p = {round(p_value2, 4)}  -> {"significant" if p_value2 < 0.05 else "not significant"}')
print()
print('TEST 2: does weather affect SPEND per order?')
print(f'  rain:        p = {round(p_value3, 4)}  -> {"significant" if p_value3 < 0.05 else "not significant"}')
print(f'  temperature: p = {round(p_value4, 4)}  -> {"significant" if p_value4 < 0.05 else "not significant"}')
print()
print('TEST 3: does weather affect CANCELLATION rate? (the guilt test)')
print(f'  rain:        p = {round(p_value5, 4)}  -> {"significant" if p_value5 < 0.05 else "not significant"}')
print(f'  temperature: p = {round(p_value6, 4)}  -> {"significant" if p_value6 < 0.05 else "not significant"}')
print()
print('(p < 0.05 = statistically significant at 95% confidence)')

HYPOTHESIS TEST RESULTS

TEST 1: does weather affect order VOLUME?
  rain:        p = 0.006  -> significant
  temperature: p = 0.7973  -> not significant

TEST 2: does weather affect SPEND per order?
  rain:        p = 0.5507  -> not significant
  temperature: p = 0.0  -> significant

TEST 3: does weather affect CANCELLATION rate? (the guilt test)
  rain:        p = 0.7492  -> not significant
  temperature: p = 0.0008  -> significant

(p < 0.05 = statistically significant at 95% confidence)
